# 01 — Dataset Ingestion & Preprocessing

**Project:** Enterprise Support Intelligence Platform

**Source dataset:** [`tobiasbueck/multilingual-customer-support-tickets`](https://www.kaggle.com/datasets/tobiasbueck/multilingual-customer-support-tickets)
(Kaggle, CC BY 4.0) — mirrored on Hugging Face as `miku1211/customer-support-tickets`.

## What this notebook does

1. Locates and loads the raw dataset (Kaggle attachment, Kaggle API, or Hugging Face fallback).
2. Validates the schema and profiles missing values and label distributions.
3. Normalizes text, categories, departments, priorities, and languages.
4. Removes exact duplicates and clusters near-duplicates.
5. Creates a duplicate-aware **16,000 / 2,000 / 2,000** split.
6. Writes standardized Parquet files and a data-quality report.

## How to run

### Kaggle (recommended)

1. Create a Kaggle notebook and attach the dataset `tobiasbueck/multilingual-customer-support-tickets`.
2. Import this notebook into the Kaggle editor (File → Import Notebook) or copy it in.
3. Run all cells. Outputs are written to `/kaggle/working/processed/`.

### Local (VS Code + Jupyter)

1. From the repository root: `uv sync --extra data --extra dev`
2. Select the project `.venv` kernel (or install the `enterprise-support` kernel — see
   `notebooks/README.md`).
3. Provide `~/.kaggle/kaggle.json`, or let the notebook fall back to the Hugging Face mirror.
4. Run all cells. Outputs are written to `data/processed/`.

> Kaggle notebooks do not expose a Jupyter server endpoint that VS Code can attach to. VS Code can
> run this notebook locally against the Kaggle API, or you can run it in the Kaggle editor after
> importing it.

## Outputs

- `tickets_retrieval.parquet` — 16,000 records, retrieval corpus only.
- `tickets_dev.parquet` — 2,000 records for prompt and retrieval tuning.
- `tickets_eval.parquet` — 2,000 held-out records for evaluation.
- `data_quality_report.json` — counts, distributions, dedup statistics, leakage check, PII scan.
- `split_manifest.json` — row counts and SHA-256 checksums.
- `metadata` — records the actual acquisition method (`kaggle_csv`, `local_csv`, or `huggingface`).

**The evaluation split is quarantined and must never be ingested into ChromaDB.**

> Phase 1 scans for PII but does not redact it. Redaction and blocking are enforced when
> records are ingested into the knowledge base and when responses pass the QA gate.

In [ ]:
from __future__ import annotations

import hashlib
import html
import json
import re
import unicodedata
import uuid
import zipfile
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RANDOM_SEED = 42
SOURCE_DATASET = "tobiasbueck/multilingual-customer-support-tickets"
SOURCE_DATASET_URL = (
    "https://www.kaggle.com/datasets/tobiasbueck/multilingual-customer-support-tickets"
)
HF_FALLBACK_DATASET = "miku1211/customer-support-tickets"

TARGET_RECORDS = 20_000
SPLIT_TARGETS = {"retrieval": 16_000, "dev": 2_000, "eval": 2_000}
TARGET_LANGUAGES: set[str] | None = {"en"}  # set to None to keep every language
MIN_QUERY_CHARS = 20
MAX_QUERY_CHARS = 20_000
NEAR_DUP_THRESHOLD = 0.88
NEAR_DUP_NEIGHBORS = 10

# Optional: pin the raw CSV if auto-detection picks the wrong file.
RAW_CSV_PATH: str | None = None

IS_KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
RAW_DIR = Path("/kaggle/input") if IS_KAGGLE else PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = Path("/kaggle/working/processed") if IS_KAGGLE else PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"Project root:      {PROJECT_ROOT}")
print(f"Raw input dir:     {RAW_DIR}")
print(f"Output dir:        {OUTPUT_DIR}")

In [ ]:
def find_csv_files(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*.csv") if p.is_file())


def extract_archives(root: Path, destination: Path) -> None:
    for archive in sorted(root.rglob("*.zip")):
        target = destination / archive.stem
        if target.exists():
            continue
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(target)
        print(f"Extracted {archive.name} -> {target}")


def pick_csv(candidates: list[Path]) -> Path:
    if RAW_CSV_PATH:
        path = Path(RAW_CSV_PATH)
        if not path.exists():
            raise FileNotFoundError(f"RAW_CSV_PATH does not exist: {path}")
        return path
    if not candidates:
        raise FileNotFoundError("No CSV files found.")
    if len(candidates) == 1:
        return candidates[0]
    preferred = [
        p
        for p in candidates
        if any(token in p.name.lower() for token in ("customer", "ticket", "support"))
    ]
    if len(preferred) == 1:
        return preferred[0]
    listing = "\n".join(f"  - {p}" for p in candidates)
    raise RuntimeError(
        "Multiple CSV files were found. Set RAW_CSV_PATH to the correct file:\n" + listing
    )


raw_candidates = find_csv_files(RAW_DIR)
extracted_dir = (
    Path("/kaggle/working/raw_extracted")
    if IS_KAGGLE
    else PROJECT_ROOT / "data" / "interim" / "raw_extracted"
)

if not raw_candidates:
    extract_archives(RAW_DIR, extracted_dir)
    raw_candidates = find_csv_files(extracted_dir)

raw_path: Path | None = None
acquisition_method = "unknown"
raw_df: pd.DataFrame | None = None
if raw_candidates:
    raw_path = pick_csv(raw_candidates)
    acquisition_method = "kaggle_csv" if IS_KAGGLE else "local_csv"
    print(f"Loading raw CSV: {raw_path}")
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            raw_df = pd.read_csv(raw_path, encoding=encoding, low_memory=False)
            print(f"Loaded {len(raw_df):,} rows with encoding={encoding}")
            break
        except UnicodeDecodeError:
            continue
    if raw_df is None:
        raise RuntimeError("Could not decode the CSV with utf-8-sig, utf-8, or latin-1.")
else:
    # Last-resort fallback: Hugging Face mirror (requires internet).
    print("No local CSV found; trying the Hugging Face mirror...")
    acquisition_method = "huggingface"
    from datasets import load_dataset

    hf_dataset = load_dataset(HF_FALLBACK_DATASET, split="train")
    raw_df = hf_dataset.to_pandas()
    print(f"Loaded {len(raw_df):,} rows from Hugging Face")

assert raw_df is not None
raw_df = raw_df.reset_index(drop=True)
raw_source_columns = [str(c) for c in raw_df.columns]
raw_df["_source_row_index"] = np.arange(len(raw_df))

print(f"Acquisition method: {acquisition_method}")
print(f"Acquisition path:   {raw_path or HF_FALLBACK_DATASET}")
display(raw_df.head(3))

In [ ]:
print(f"Shape: {raw_df.shape}")
print("\nColumns:")
print(list(raw_df.columns))

print("\nDtypes:")
display(raw_df.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = raw_df.isna().mean().sort_values(ascending=False).rename("missing_rate")
display(missing.to_frame())

print("\nUnique values (top 25):")
display(raw_df.nunique().sort_values(ascending=False).head(25).to_frame("n_unique"))

for column in ("language", "type", "queue", "priority"):
    if column in raw_df.columns:
        print(f"\nValue counts for `{column}`:")
        display(raw_df[column].value_counts(dropna=False).head(20).to_frame("count"))

In [ ]:
COLUMN_ALIASES: dict[str, tuple[str, ...]] = {
    "subject": ("subject", "title", "ticket_subject"),
    "customer_query": ("body", "description", "query", "message", "ticket_description"),
    "agent_response": ("answer", "response", "resolution", "agent_response"),
    "category": ("type", "category", "ticket_type"),
    "department": ("queue", "department", "team"),
    "priority": ("priority", "urgency"),
    "language": ("language", "lang"),
}


def resolve_columns(df: pd.DataFrame) -> dict[str, str | None]:
    normalized = {c.strip().lower().replace(" ", "_"): c for c in df.columns}
    resolved: dict[str, str | None] = {}
    for target, aliases in COLUMN_ALIASES.items():
        resolved[target] = next((normalized[a] for a in aliases if a in normalized), None)
    return resolved


resolved = resolve_columns(raw_df)
print("Resolved columns:")
for key, value in resolved.items():
    print(f"  {key:16s} -> {value}")

required = ["customer_query"]
missing_required = [c for c in required if resolved.get(c) is None]
if missing_required:
    raise ValueError(f"Missing required source columns: {missing_required}")

In [ ]:
def clean_text(value: Any) -> str:
    if value is None or value is pd.NA:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    text = html.unescape(str(value))
    text = unicodedata.normalize("NFKC", text)
    text = "".join(
        ch for ch in text if ch in {"\n", "\t"} or not unicodedata.category(ch).startswith("C")
    )
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_label(value: Any) -> str:
    return re.sub(r"\s+", " ", clean_text(value)).strip().lower()


PRIORITY_MAP = {
    "low": "low",
    "1": "low",
    "p1": "low",
    "medium": "medium",
    "normal": "medium",
    "moderate": "medium",
    "2": "medium",
    "p2": "medium",
    "high": "high",
    "3": "high",
    "p3": "high",
    "urgent": "high",
    "critical": "critical",
    "4": "critical",
    "p4": "critical",
    "severe": "critical",
}

CATEGORY_MAP = {
    "incident": "Incident",
    "request": "Request",
    "problem": "Problem",
    "change": "Change",
    "question": "Request",
    "inquiry": "Request",
    "bug": "Problem",
}

DEPARTMENT_MAP = {
    "technical support": "Technical Support",
    "customer service": "Customer Service",
    "billing and payments": "Billing and Payments",
    "product support": "Product Support",
    "it support": "IT Support",
    "returns and exchanges": "Returns and Exchanges",
    "sales and pre-sales": "Sales and Pre-Sales",
    "human resources": "Human Resources",
    "service outages and maintenance": "Service Outages and Maintenance",
    "general inquiry": "General Inquiry",
}

LANGUAGE_MAP = {
    "en": "en",
    "english": "en",
    "de": "de",
    "german": "de",
    "deutsch": "de",
    "es": "es",
    "spanish": "es",
    "español": "es",
    "fr": "fr",
    "french": "fr",
    "français": "fr",
    "pt": "pt",
    "portuguese": "pt",
    "português": "pt",
}


def normalize_priority(value: Any) -> str:
    key = normalize_label(value)
    if key in PRIORITY_MAP:
        return PRIORITY_MAP[key]
    return key if key in {"low", "medium", "high", "critical"} else "unknown"


def normalize_category(value: Any) -> str:
    key = normalize_label(value)
    if not key:
        return "Unknown"
    return CATEGORY_MAP.get(key, clean_text(value).title())


def normalize_department(value: Any) -> str:
    key = normalize_label(value)
    if not key:
        return "Unknown"
    return DEPARTMENT_MAP.get(key, clean_text(value).title())


def normalize_language(value: Any) -> str:
    key = normalize_label(value)
    if not key:
        return "unknown"
    return LANGUAGE_MAP.get(key, key)

In [ ]:
tag_columns = [c for c in raw_df.columns if re.fullmatch(r"tag_\d+", str(c))]


def dedupe_tags(values: list[str]) -> list[str]:
    seen: set[str] = set()
    result: list[str] = []
    for value in values:
        key = value.casefold()
        if value and key not in seen:
            seen.add(key)
            result.append(value)
    return result


std = pd.DataFrame(index=raw_df.index)
std["_source_row_index"] = raw_df["_source_row_index"].astype(int)
std["subject"] = (
    raw_df[resolved["subject"]].map(clean_text) if resolved["subject"] else "(no subject)"
)
std["customer_query"] = raw_df[resolved["customer_query"]].map(clean_text)
std["agent_response"] = (
    raw_df[resolved["agent_response"]].map(clean_text) if resolved["agent_response"] else ""
)
std["category"] = (
    raw_df[resolved["category"]].map(normalize_category) if resolved["category"] else "Unknown"
)
std["department"] = (
    raw_df[resolved["department"]].map(normalize_department)
    if resolved["department"]
    else "Unknown"
)
std["priority"] = (
    raw_df[resolved["priority"]].map(normalize_priority) if resolved["priority"] else "unknown"
)
std["language"] = (
    raw_df[resolved["language"]].map(normalize_language) if resolved["language"] else "unknown"
)
if tag_columns:
    std["tags"] = raw_df[tag_columns].apply(
        lambda row: dedupe_tags([clean_text(row[c]) for c in tag_columns]), axis=1
    )
else:
    std["tags"] = [[] for _ in range(len(std))]
std["business_type"] = (
    raw_df["business_type"].map(clean_text) if "business_type" in raw_df.columns else ""
)
std["source_version"] = (
    raw_df["version"].map(lambda value: "" if pd.isna(value) else str(value).strip())
    if "version" in raw_df.columns
    else ""
)

std["subject"] = std["subject"].mask(std["subject"] == "", "(no subject)")
std["customer_query"] = std["customer_query"].mask(std["customer_query"] == "", np.nan)

print(f"Rows before cleaning: {len(std):,}")
print(f"Empty queries:        {int(std['customer_query'].isna().sum()):,}")
print(f"Empty answers:        {int((std['agent_response'] == '').sum()):,}")

In [ ]:
cleaning_report: dict[str, int] = {}
cleaning_report["raw_rows"] = len(std)

# Drop rows without a usable customer query.
before = len(std)
std = std[std["customer_query"].notna()].copy()
cleaning_report["dropped_empty_query"] = before - len(std)

# The retrieval corpus needs a historical resolution as evidence.
before = len(std)
std = std[std["agent_response"].str.len() > 0].copy()
cleaning_report["dropped_empty_answer"] = before - len(std)

# Enforce query length bounds.
before = len(std)
query_lengths = std["customer_query"].str.len()
std = std[(query_lengths >= MIN_QUERY_CHARS) & (query_lengths <= MAX_QUERY_CHARS)].copy()
cleaning_report["dropped_query_length"] = before - len(std)

# Language filter.
if TARGET_LANGUAGES:
    before = len(std)
    std = std[std["language"].isin(TARGET_LANGUAGES)].copy()
    cleaning_report["dropped_language"] = before - len(std)
else:
    cleaning_report["dropped_language"] = 0

std = std.reset_index(drop=True)
cleaning_report["rows_before_exact_dedupe"] = len(std)

print("Cleaning report:")
for key, value in cleaning_report.items():
    print(f"  {key:28s} {value:,}")
print(f"\nRows before exact dedupe: {len(std):,}")

if TARGET_LANGUAGES and len(std) < TARGET_RECORDS:
    print(
        f"WARNING: only {len(std):,} rows remain after the language filter "
        f"(target: {TARGET_RECORDS:,}). Set TARGET_LANGUAGES = None or add languages."
    )

display(std["language"].value_counts().to_frame("count"))
display(std["category"].value_counts().to_frame("count"))
display(std["priority"].value_counts().to_frame("count"))
display(std["department"].value_counts().to_frame("count"))

In [ ]:
def fingerprint(*parts: str) -> str:
    payload = "\x1f".join(parts)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


std["_exact_key"] = [
    fingerprint(subject.lower().strip(), query.lower().strip(), answer.lower().strip())
    for subject, query, answer in zip(
        std["subject"], std["customer_query"], std["agent_response"], strict=True
    )
]

before = len(std)
std = std.drop_duplicates("_exact_key", keep="first").reset_index(drop=True)
cleaning_report["dropped_exact_duplicates"] = before - len(std)
cleaning_report["rows_after_cleaning"] = len(std)

print(f"Exact duplicates removed: {cleaning_report['dropped_exact_duplicates']:,}")
print(f"Rows after cleaning:      {len(std):,}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

BOILERPLATE_PATTERNS = [
    r"dear (customer )?support( team)?[,:]?",
    r"dear customer service[,:]?",
    r"i hope this (message|email) (finds|reaches) you well[.!]?",
    r"i hope you are doing well[.!]?",
    r"best regards[,:]?",
    r"kind regards[,:]?",
    r"sincerely[,:]?",
    r"thank you for your (assistance|help|time)[.!]?",
    r"<name>",
    r"<tel_num>",
]


def similarity_text(subject: str, query: str) -> str:
    text = f"{subject} {query}".lower()
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


similarity_corpus = [
    similarity_text(subject, query)
    for subject, query in zip(std["subject"], std["customer_query"], strict=True)
]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=60_000,
    sublinear_tf=True,
    strip_accents="unicode",
)
tfidf = vectorizer.fit_transform(similarity_corpus)
print(f"TF-IDF matrix: {tfidf.shape[0]:,} documents x {tfidf.shape[1]:,} features")

n_neighbors = min(NEAR_DUP_NEIGHBORS, len(std))
nn = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute", n_jobs=-1)
nn.fit(tfidf)
distances, indices = nn.kneighbors(tfidf)

parent = list(range(len(std)))
rank = [0] * len(std)


def find(x: int) -> int:
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a: int, b: int) -> None:
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    if rank[ra] == rank[rb]:
        rank[ra] += 1


pair_links = 0
near_dup_pairs: set[tuple[int, int]] = set()
for i in range(len(std)):
    for distance, j in zip(distances[i], indices[i], strict=True):
        j = int(j)
        if i == j:
            continue
        if (1.0 - float(distance)) >= NEAR_DUP_THRESHOLD:
            union(i, j)
            pair_links += 1
            near_dup_pairs.add((min(i, j), max(i, j)))

pair_count = len(near_dup_pairs)
print(f"Near-duplicate neighbor links (similarity >= {NEAR_DUP_THRESHOLD}): {pair_links:,}")
print(f"Near-duplicate unique pairs (similarity >= {NEAR_DUP_THRESHOLD}): {pair_count:,}")

std["_cluster_id"] = [find(i) for i in range(len(std))]
cluster_sizes = std.groupby("_cluster_id").size()
print(f"Near-duplicate clusters: {len(cluster_sizes):,}")
print(f"Records in multi-record clusters: {int(cluster_sizes[cluster_sizes > 1].sum()):,}")
print(f"Largest cluster: {int(cluster_sizes.max()):,} records")

if int(cluster_sizes.max()) > max(10, int(len(std) * 0.05)):
    print(
        "WARNING: the largest cluster exceeds 5% of records. Consider raising NEAR_DUP_THRESHOLD."
    )

display(cluster_sizes.sort_values(ascending=False).head(10).to_frame("cluster_size"))

In [ ]:
def fill_split(pool: pd.DataFrame, target: int, seed: int) -> tuple[set[int], int]:
    sizes = pool.groupby("_cluster_id").size().to_dict()
    rng = np.random.default_rng(seed)
    clusters = list(sizes.items())
    rng.shuffle(clusters)
    clusters.sort(key=lambda item: item[1], reverse=True)

    chosen: list[int] = []
    total = 0
    for cluster_id, size in clusters:
        if total + size <= target:
            chosen.append(cluster_id)
            total += size
        if total == target:
            break

    if total < target:
        chosen_set = set(chosen)
        singles = [cid for cid, size in sizes.items() if size == 1 and cid not in chosen_set]
        rng.shuffle(singles)
        need = target - total
        chosen.extend(singles[:need])
        total += min(need, len(singles))

    return set(chosen), total


# Carve out eval first, then dev, then retrieval. Whole near-duplicate clusters
# are assigned to exactly one split; clusters that do not fit are left unused.
eval_clusters, eval_count = fill_split(std, SPLIT_TARGETS["eval"], RANDOM_SEED)
remaining = std[~std["_cluster_id"].isin(eval_clusters)].copy()

dev_clusters, dev_count = fill_split(remaining, SPLIT_TARGETS["dev"], RANDOM_SEED + 1)
remaining = remaining[~remaining["_cluster_id"].isin(dev_clusters)].copy()

retrieval_clusters, retrieval_count = fill_split(
    remaining, SPLIT_TARGETS["retrieval"], RANDOM_SEED + 2
)

std["split"] = "unused"
std.loc[std["_cluster_id"].isin(eval_clusters), "split"] = "eval"
std.loc[std["_cluster_id"].isin(dev_clusters), "split"] = "dev"
std.loc[std["_cluster_id"].isin(retrieval_clusters), "split"] = "retrieval"

selected = std[std["split"] != "unused"].copy()
dropped_unused = int((std["split"] == "unused").sum())

print("Split targets vs actual:")
for name in ("retrieval", "dev", "eval"):
    actual = int((selected["split"] == name).sum())
    print(f"  {name:10s} target={SPLIT_TARGETS[name]:>6,}  actual={actual:>6,}")
print(f"  {'unused':10s} target={0:>6,}  actual={dropped_unused:>6,}")
print(f"  {'total':10s} target={sum(SPLIT_TARGETS.values()):>6,}  actual={len(selected):>6,}")

if len(selected) < sum(SPLIT_TARGETS.values()):
    print(
        f"WARNING: only {len(selected):,} records were selected. "
        "This usually means near-duplicate clusters were too large to fit the targets."
    )

In [ ]:
split_order = {"retrieval": 0, "dev": 1, "eval": 2}
selected["_split_order"] = selected["split"].map(split_order)
selected = selected.sort_values(["_split_order", "_cluster_id", "_source_row_index"]).reset_index(
    drop=True
)


def make_ticket_id(source_row_index: int) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{SOURCE_DATASET}:{source_row_index}"))


selected["ticket_id"] = selected["_source_row_index"].map(make_ticket_id)
selected["source_ticket_id"] = selected["_source_row_index"].map(lambda i: f"{SOURCE_DATASET}:{i}")
selected["resolution_status"] = np.where(
    selected["agent_response"].str.len() > 0, "resolved", "unresolved"
)
selected["source_dataset"] = SOURCE_DATASET
selected["created_at"] = pd.Series(pd.NaT, index=selected.index, dtype="datetime64[ns]")
selected["metadata"] = selected.apply(
    lambda row: json.dumps(
        {
            "source_system": "kaggle",
            "source_dataset": SOURCE_DATASET,
            "source_dataset_url": SOURCE_DATASET_URL,
            "source_has_native_id": False,
            "source_ticket_id_kind": "derived_locator",
            "acquisition_method": acquisition_method,
            "acquisition_path": str(raw_path) if raw_path else HF_FALLBACK_DATASET,
            "source_row_index": int(row["_source_row_index"]),
            "source_version": row["source_version"],
            "business_type": row["business_type"],
            "near_dup_cluster_id": int(row["_cluster_id"]),
            "near_dup_cluster_size": int(cluster_sizes.loc[row["_cluster_id"]]),
            "split": row["split"],
            "has_source_timestamp": False,
        },
        ensure_ascii=False,
    ),
    axis=1,
)
selected["tags"] = selected["tags"].map(lambda tags: json.dumps(tags, ensure_ascii=False))

FINAL_COLUMNS = [
    "ticket_id",
    "source_ticket_id",
    "subject",
    "customer_query",
    "agent_response",
    "category",
    "department",
    "priority",
    "language",
    "tags",
    "resolution_status",
    "source_dataset",
    "created_at",
    "metadata",
]
final_df = selected[FINAL_COLUMNS].copy()

display(final_df.head(3))
print(f"Final rows: {len(final_df):,}")
print(f"Columns: {list(final_df.columns)}")

In [ ]:
split_sets = {
    name: set(selected.loc[selected["split"] == name, "ticket_id"])
    for name in ("retrieval", "dev", "eval")
}
overlaps = {
    "retrieval_eval": len(split_sets["retrieval"] & split_sets["eval"]),
    "retrieval_dev": len(split_sets["retrieval"] & split_sets["dev"]),
    "dev_eval": len(split_sets["dev"] & split_sets["eval"]),
}
assert all(value == 0 for value in overlaps.values()), overlaps

cluster_split_counts = selected.groupby("_cluster_id")["split"].nunique()
leakage_clusters = int((cluster_split_counts > 1).sum())
assert leakage_clusters == 0, f"{leakage_clusters} clusters span multiple splits"

print("Leakage checks passed.")
print(f"  Split overlaps: {overlaps}")
print(f"  Clusters spanning splits: {leakage_clusters}")

PII_PATTERNS = {
    "email": re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"),
    "phone": re.compile(r"\+?\d[\d\s\-()]{7,}\d"),
    "url": re.compile(r"https?://\S+"),
}


def count_pii(series: pd.Series) -> dict[str, int]:
    return {
        name: int(series.str.contains(pattern, regex=True, na=False).sum())
        for name, pattern in PII_PATTERNS.items()
    }


pii_report = {
    "customer_query": count_pii(final_df["customer_query"]),
    "agent_response": count_pii(final_df["agent_response"]),
}
print("PII pattern counts (rows):")
display(pd.DataFrame(pii_report))


def value_counts_dict(series: pd.Series) -> dict[str, int]:
    return {str(key): int(value) for key, value in series.value_counts(dropna=False).items()}


report = {
    "generated_at": datetime.now(UTC).isoformat(),
    "source": {
        "dataset": SOURCE_DATASET,
        "url": SOURCE_DATASET_URL,
        "hf_fallback": HF_FALLBACK_DATASET,
        "acquisition_method": acquisition_method,
        "acquisition_path": str(raw_path) if raw_path else HF_FALLBACK_DATASET,
        "has_native_ticket_id": False,
        "raw_rows": len(raw_df),
        "raw_columns": raw_source_columns,
    },
    "configuration": {
        "target_records": TARGET_RECORDS,
        "split_targets": SPLIT_TARGETS,
        "target_languages": sorted(TARGET_LANGUAGES) if TARGET_LANGUAGES else None,
        "min_query_chars": MIN_QUERY_CHARS,
        "max_query_chars": MAX_QUERY_CHARS,
        "near_dup_threshold": NEAR_DUP_THRESHOLD,
        "random_seed": RANDOM_SEED,
    },
    "cleaning": cleaning_report,
    "final": {
        "available_rows": len(std),
        "rows": len(final_df),
        "splits": {
            name: int((selected["split"] == name).sum()) for name in ("retrieval", "dev", "eval")
        },
        "unused_rows": dropped_unused,
        "columns": list(final_df.columns),
    },
    "distributions": {
        "language": value_counts_dict(final_df["language"]),
        "category": value_counts_dict(final_df["category"]),
        "priority": value_counts_dict(final_df["priority"]),
        "department": value_counts_dict(final_df["department"]),
        "resolution_status": value_counts_dict(final_df["resolution_status"]),
    },
    "near_duplicates": {
        "clusters": len(cluster_sizes),
        "multi_record_clusters": int((cluster_sizes > 1).sum()),
        "records_in_multi_record_clusters": int(cluster_sizes[cluster_sizes > 1].sum()),
        "largest_cluster": int(cluster_sizes.max()),
    },
    "leakage": {
        "split_overlaps": overlaps,
        "clusters_spanning_splits": leakage_clusters,
        "passed": all(value == 0 for value in overlaps.values()) and leakage_clusters == 0,
    },
    "pii_scan": pii_report,
    "pii_policy": {
        "scanned_patterns": sorted(PII_PATTERNS),
        "redacted": False,
        "note": (
            "Phase 1 only scans for PII. Redaction/blocking is enforced when records are "
            "ingested into the knowledge base and when responses pass the QA gate."
        ),
    },
}

In [ ]:
manifest: dict[str, Any] = {
    "generated_at": report["generated_at"],
    "source": {
        "dataset": SOURCE_DATASET,
        "acquisition_method": acquisition_method,
        "acquisition_path": str(raw_path) if raw_path else HF_FALLBACK_DATASET,
    },
    "files": {},
}

for split_name in ("retrieval", "dev", "eval"):
    split_df = selected.loc[selected["split"] == split_name, FINAL_COLUMNS].reset_index(drop=True)
    path = OUTPUT_DIR / f"tickets_{split_name}.parquet"
    split_df.to_parquet(path, index=False)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest["files"][split_name] = {
        "path": path.name,
        "rows": len(split_df),
        "sha256": digest,
    }
    print(f"Wrote {path} ({len(split_df):,} rows)")

report_path = OUTPUT_DIR / "data_quality_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {report_path}")

manifest_path = OUTPUT_DIR / "split_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {manifest_path}")

# On Kaggle, bundle the outputs for a single download.
if IS_KAGGLE:
    archive_path = Path("/kaggle/working/processed_tickets.zip")
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file in OUTPUT_DIR.glob("*"):
            zf.write(file, arcname=file.name)
    print(f"Wrote {archive_path}")

In [ ]:
summary = pd.DataFrame(
    {
        "split": ["retrieval", "dev", "eval"],
        "rows": [report["final"]["splits"][name] for name in ("retrieval", "dev", "eval")],
        "purpose": [
            "historical retrieval corpus",
            "prompt / retrieval tuning",
            "held-out evaluation",
        ],
    }
)
display(summary)

print("Top categories in the retrieval corpus:")
display(
    final_df.loc[selected["split"].eq("retrieval"), "category"]
    .value_counts()
    .head(10)
    .to_frame("count")
)

print("Sample retrieval record:")
sample = final_df.loc[selected["split"].eq("retrieval")].iloc[0]
for field in (
    "ticket_id",
    "subject",
    "customer_query",
    "agent_response",
    "category",
    "priority",
):
    print(f"\n{field}:\n{str(sample[field])[:800]}")

In [ ]:
# Phase 1 verification snapshot. Copy the block between the START/END markers.
verification = {
    "source": report["source"],
    "configuration": report["configuration"],
    "cleaning": cleaning_report,
    "rows": {
        "raw": len(raw_df),
        "before_exact_dedupe": cleaning_report.get("rows_before_exact_dedupe"),
        "after_cleaning": cleaning_report.get("rows_after_cleaning"),
        "final": len(final_df),
        "unused": dropped_unused,
    },
    "splits": {
        "targets": SPLIT_TARGETS,
        "actual": {
            name: int((selected["split"] == name).sum()) for name in ("retrieval", "dev", "eval")
        },
    },
    "leakage": {
        "split_overlaps": overlaps,
        "clusters_spanning_splits": leakage_clusters,
        "passed": report["leakage"]["passed"],
    },
    "near_duplicates": report["near_duplicates"],
    "distributions": report["distributions"],
    "pii": pii_report,
    "outputs": {
        path.name: {"size_mb": round(path.stat().st_size / 1_048_576, 2)}
        for path in sorted(OUTPUT_DIR.glob("*"))
        if path.is_file()
    },
    "checksums": {name: info["sha256"] for name, info in manifest["files"].items()},
    "warnings": [],
}

if len(selected) < sum(SPLIT_TARGETS.values()):
    verification["warnings"].append("Selected fewer rows than the split targets.")
if not report["leakage"]["passed"]:
    verification["warnings"].append("Leakage check failed.")

print("=== PHASE 1 VERIFICATION START ===")
print(json.dumps(verification, indent=2, ensure_ascii=False, default=str))
print("=== PHASE 1 VERIFICATION END ===")

## Next steps

- Ingest `tickets_retrieval.parquet` into ChromaDB in the retrieval notebook.
- Keep `tickets_eval.parquet` out of the vector store.
- Use `tickets_dev.parquet` for prompt and retrieval tuning.
- Record the `split_manifest.json` checksums alongside the evaluation report for reproducibility.